# 부품교체이력 EDA · 소둔로 관련 항목

- 담당: 정수진 · Issue #50
- **목적:** 컬럼의 의미를 이해하고, 어떤 부품을 왜 교체했는지 확인한다.
- **소둔로 확인:** 정비이력으로 대상 설비를 찾고 부품마스터로 부품명을 확인한다.
- 위에서부터 실행한다. 개별 작업 표는 로컬 확인용이며 공개 공유·커밋 전 출력과 실행번호를 제거한다.

## 1. 컬럼은 무엇을 뜻하는가?

| 컬럼 | 의미 | 알 수 있는 것 | 알 수 없는 것 |
| --- | --- | --- | --- |
| WO_NO | 정비 작업번호 | 어느 작업에 속한 교체 내역인지 | 설비·날짜는 정비이력 연결 필요 |
| PART_CD | 부품코드 | 어떤 코드의 부품을 교체했는지 | 부품명·기능은 부품마스터 연결 필요 |
| QTY | 해당 행의 교체수량 | 기록된 수량 | 개/세트 단위, 잔여재고·잔여수명 |
| CHG_RSN | 교체 사유 | 손상·수명도래·예방 구분 | 실제 고장 원인·교체 효과 |

- **한 행:** 한 작업에 기록된 특정 부품의 수량·교체 사유 내역이다.
- **QTY=3:** 기록 수량이 3이라는 뜻이며, 3회 교체라는 뜻이 아니다.

In [ ]:
import os
import pandas as pd
from IPython.display import display

# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
data_folder = "data"
if not os.path.isdir(data_folder):
    data_folder = "../data"

df = pd.read_csv(
    data_folder + "/G-02_부품교체이력.csv",
    encoding="utf-8-sig", dtype="string")

## 2. 데이터 규모와 품질

**확인 목적:** 작업 수와 교체 내역 수를 구분하고, 빈값·중복·수량 문제를 확인한다.

In [ ]:
# 같은 작업번호는 한 번만 셉니다.
size = pd.Series({
    "교체 내역 행 수(전체 행)": len(df),
    "고유 정비 작업 수(WO_NO)": df["WO_NO"].nunique(),
    "부품 종류 수(PART_CD)": df["PART_CD"].nunique(),
    "작업-부품 조합 수(WO_NO + PART_CD)": len(df[["WO_NO", "PART_CD"]].drop_duplicates()),
})
display(size.to_frame("개수"))

# 컬럼별 자료형과 결측을 확인합니다.
quality = pd.DataFrame()
quality["읽은 자료형"] = df.dtypes.astype(str)
quality["결측 수"] = df.isna().sum()
for column in df.columns:
    quality.loc[column, "공백 수"] = (df[column].str.strip() == "").sum()
display(quality)

In [ ]:
# 원본은 바꾸지 않고 수량을 숫자로 읽어 봅니다.
qty = pd.to_numeric(df["QTY"], errors="coerce")
checks = pd.Series({
    "완전 중복 추가 행": df.duplicated().sum(),
    "WO_NO + PART_CD 반복 추가 행": df.duplicated(["WO_NO", "PART_CD"]).sum(),
    "QTY 숫자 변환 실패": (df["QTY"].notna() & qty.isna()).sum(),
    "QTY가 0인 행": (qty == 0).sum(),
    "QTY가 음수인 행": (qty < 0).sum(),
    "QTY가 소수인 행": (qty.notna() & (qty % 1 != 0)).sum(),
})
display(checks.to_frame("행 수"))

### 결과

- **427행 · 380작업(WO_NO) · 71종(PART_CD):** 한 작업에 여러 교체 내역이 있다.
- 결측·공백·수량 형식 문제는 없다.
- 완전 중복 후보 4행은 작업-부품 반복 13행에 포함된다. 원인 확인 전 삭제하지 않는다.

## 3. 어떤 부품을 왜 교체했는가?

**확인 목적:** 전체 공정의 부품별 고유 작업 수와 교체 사유를 본다. 교체 빈도는 고장률이 아니다.

In [ ]:
# 정비 작업 수 기준 TOP 5입니다.
part_count = df.groupby("PART_CD")["WO_NO"].nunique()
top5 = part_count.sort_values(ascending=False).head(5)
display(top5.to_frame("고유 정비 작업 수(WO_NO)"))

# 행 수와 작업 수는 구분해서 표시합니다.
reason = pd.DataFrame()
reason["교체 내역 행 수"] = df["CHG_RSN"].value_counts()
reason["행 비율(%)"] = (reason["교체 내역 행 수"] / len(df) * 100).round(1)
reason["고유 정비 작업 수(WO_NO)"] = df.groupby("CHG_RSN")["WO_NO"].nunique()
display(reason)

# 한 작업에 여러 사유가 있을 수 있습니다.
reason_count = df.groupby("WO_NO")["CHG_RSN"].nunique()
print("여러 사유가 기록된 작업 수:", (reason_count > 1).sum())

### 결과

- **작업 수 TOP 5:** SEL-009 35 → SEL-006 28 → SEL-003 22 → SEL-001 19 → CWP-002 17.
- **사유:** 손상 37.2% · 수명도래 34.9% · 예방 27.9%. 교체 전체가 고장은 아니다.
- 24작업에 여러 사유가 있어 사유별 작업 수는 서로 겹친다. 위 순위는 전체 공정 기준이다.

## 4. 소둔로 관련 내역 찾기

**연결:** 교체이력 WO_NO → 정비이력 EQP_TAG → 소둔로 P1-CR1-CAL01 선택. PART_CD → 부품마스터 PART_NM.

- 설비코드를 먼저 제한한다. 부품코드만 고르면 다른 공정의 같은 부품도 섞인다.
- 이번 범위는 소둔로 본체 코드와 정확히 일치하는 작업이다. 하위 설비 포함 여부는 별도 검토한다.

In [ ]:
# 필요한 컬럼만 읽습니다.
maintenance = pd.read_csv(
    data_folder + "/G-02_정비이력.csv", encoding="utf-8-sig", dtype="string",
    usecols=["WO_NO", "EQP_TAG", "STRT_DT"])
parts = pd.read_csv(
    data_folder + "/G-01_부품마스터.csv", encoding="utf-8-sig", dtype="string",
    usecols=["PART_CD", "PART_NM"])

# 대상 키에 빈값이나 중복이 있으면 연결 전에 확인합니다.
for table, key in [(df, "WO_NO"), (df, "PART_CD"),
                   (maintenance, "WO_NO"), (parts, "PART_CD")]:
    assert table[key].notna().all(), "연결 키에 결측이 있습니다."
    assert table[key].str.strip().ne("").all(), "연결 키에 공백이 있습니다."
assert maintenance["WO_NO"].is_unique, "정비 작업번호가 중복됩니다."
assert parts["PART_CD"].is_unique, "부품코드가 중복됩니다."

print("정비 미매칭 행:", (~df["WO_NO"].isin(maintenance["WO_NO"])).sum())
print("부품 미매칭 행:", (~df["PART_CD"].isin(parts["PART_CD"])).sum())
linked = df.merge(maintenance, on="WO_NO", how="left", validate="many_to_one")
linked = linked.merge(parts, on="PART_CD", how="left", validate="many_to_one")
print("연결 전 / 후 행 수:", len(df), "/", len(linked))

# ✅ 소둔로 관련 내용으로 확인하면 좋음

In [ ]:
# 소둔로 교체 내역만 선택합니다.
cal = linked[linked["EQP_TAG"] == "P1-CR1-CAL01"].copy()
cal = cal.sort_values(["STRT_DT", "WO_NO", "PART_CD"])

# 개별 작업 표는 로컬 확인용입니다. 공개 공유 전 출력을 제거하세요.
columns = ["WO_NO", "PART_CD", "PART_NM", "QTY", "CHG_RSN", "STRT_DT"]
display(cal[columns])

print("소둔로 교체 내역 행 수:", len(cal))
print("소둔로 고유 정비 작업 수(WO_NO):", cal["WO_NO"].nunique())
print("소둔로 부품 종류 수(PART_CD):", cal["PART_CD"].nunique())

In [ ]:
# 부품별 작업 수와 사유별 내역 행 수를 한 표로 봅니다.
cal_summary = cal.groupby(["PART_CD", "PART_NM"])["WO_NO"].nunique()
cal_summary = cal_summary.to_frame("고유 작업 수(WO_NO)")

cal_reason = pd.crosstab([cal["PART_CD"], cal["PART_NM"]], cal["CHG_RSN"])
cal_reason = cal_reason.add_suffix(" 내역 행 수")
cal_summary = cal_summary.join(cal_reason)
display(cal_summary)

### 소둔로 결과

| PART_CD | PART_NM | 고유 작업 수(WO_NO) | 손상 / 수명도래 / 예방 내역 행 수 |
| --- | --- | ---: | --- |
| INS-006 | 가스 분석계 셀 | 3 | 1 / 1 / 1 |
| SEL-006 | 오일씰 150x190x18 | 3 | 2 / 1 / 0 |
| SEL-009 | O링 세트 | 3 | 1 / 1 / 1 |

- **3종 · 8작업 · 9행.** 같은 작업에 여러 부품이 포함된다.
- 미매칭 0행, 연결 전후 427행 유지. SEL-006의 손상 2행은 상세 확인 후보다.
- QTY 단위와 장착 위치는 미확인. STRT_DT는 정비 시작 시각이다.

## 5. 최종 요약

| 확인 항목 | 결과 |  |
| --- | --- | --- |
| 데이터 규모 | 427행 · 고유 WO_NO 380개 · PART_CD 71종 | 한 작업에 여러 부품 교체 내역이 포함됨 |
| 데이터 품질 | 결측·공백·수량 형식 확인 후보 0행 | 기본 집계 가능. 기록 내용의 정확성은 별도 확인 필요 |
| 반복 기록 | 완전 중복 추가 4행, WO_NO·PART_CD 반복 추가 13행 | 4행은 13행에 포함. 실제 중복 입력 여부는 미확인 |
| 전체 공정 교체 빈도 | SEL-009가 고유 WO_NO 35개로 최다 | 상세 검토 후보이며 고장률 순위는 아님 |
| 교체 사유 | 손상 37.2% · 수명도래 34.9% · 예방 27.9% (행 기준) | 모든 교체가 고장은 아님. 24작업은 여러 사유 포함 |
| 정비·부품 연결 | 미매칭 0행, 연결 전후 427행 | WO_NO·PART_CD로 설비·정비 시각·부품명 연결 가능 |
| 소둔로 대상 | P1-CR1-CAL01: 3종 · 8작업 · 9행 | INS-006·SEL-006·SEL-009 교체 기록 확인 |
| 소둔로 확인 후보 | SEL-006의 손상 내역 2행 | 정비 상세와 관련 센서를 확인할 후보 |
| 판단할 수 없는 것 | QTY 단위, 실제 중복 원인, 고장률·수명·교체 효과 | 추가 정보 없이 확정할 수 없음 |

## 6. 다음 연결

| 현재 컬럼 | 연결할 데이터·컬럼 |
| --- | --- |
| WO_NO | 정비이력 WO_NO → DETC_DT·STRT_DT·END_DT·FAIL_MODE |
| PART_CD | 부품마스터 PART_CD → PART_NM·PART_TYPE·APPLY_EQP_TYPE |
| 정비이력 EQP_TAG | 설비마스터 EQP_TAG → PARENT_TAG·PLANT_CD |
| 소둔로 정비 시각 | 온도 MEAS_DT·LOT_NO·관련 TC/OP (시간 구간 대조) |
| 설비마스터 PLANT_CD + 시각 | 조업이벤트 PLANT_CD·EVT_DT, 정기수리캘린더 PLANT_CD·STRT_DT·END_DT |
| PART_CD·WO_NO | 추가 확보할 자재 단위·장착 위치·정비 상세 |

- 개별 작업 표는 로컬 확인용이다. 공개 공유·커밋 전 출력·실행번호를 제거한다.